In [1]:
import os

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\Data analysis\\Kaggle_Competition\\Notebook'

In [3]:
os.chdir('../')

In [18]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\Data analysis\\Kaggle_Competition'

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    inventory: Path
    product: Path
    promotion: Path
    sale_q1: Path
    sale_q2: Path
    sale_q3: Path
    sale_q4: Path
    inventory_t: Path
    products_t: Path
    promotion_t: Path
    sales_t: Path

In [21]:
from etl.constants import *
from etl.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH):
        
        self.config = read_yaml(config_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            inventory=config.inventory,
            product=config.product,
            promotion=config.promotion,
            sale_q1=config.sale_q1,
            sale_q2=config.sale_q2,
            sale_q3=config.sale_q3,
            sale_q4=config.sale_q4,
            inventory_t=config.inventory_t,
            products_t=config.products_t,
            promotion_t=config.promotion_t,
            sales_t=config.sales_t       
            )

        return data_transformation_config

In [7]:
from etl import logger
import pandas as pd
import re

In [24]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def common_cleaning(self, row):
        if row['size'] == 'Each':
            row['size'] = '1 ct'
        
        row['size'] = self.extract_numeric(row['size'])
        
        return row
    
    def extract_numeric(self, value):
        if isinstance(value, str):
            value = value.strip()
            
            #Fraction
            if re.search(r'\d+/\d+', value):
                frac = re.search(r'(\d+)/(\d+)', value)
                return float(frac.group(1)) / float(frac.group(2))

            num = re.search(r'\d+\.?\d*', value)
            if num:
                return float(num.group())
        
        return None

    def load_data(self):
        inventory = pd.read_csv(self.config.inventory)
        product = pd.read_csv(self.config.product)
        promotion = pd.read_csv(self.config.promotion)
        sale_q1 = pd.read_csv(self.config.sale_q1)
        sale_q2 = pd.read_csv(self.config.sale_q2)
        sale_q3 = pd.read_csv(self.config.sale_q3)
        sale_q4 = pd.read_csv(self.config.sale_q4)

        sales = pd.concat([sale_q1,sale_q2,sale_q3,sale_q4])
        logger.info('Sales is concat Successful.....')
        print(sales.shape)

        sales['promo_type'].fillna('No Promo', inplace=True)
        sales['promo_id'].fillna('No Promo', inplace=True)
        logger.info('Sales filled the null values.......')

        sales['sale_datetime'] = pd.to_datetime(sales['sale_datetime'], format='mixed')
        logger.info('Sales datetime convert sucessful.......')

        sales = sales.apply(self.common_cleaning, axis=1)
        logger.info('Sales Common cleaning done.........')



        promotion['start_date'] = pd.to_datetime(promotion['start_date'])
        logger.info('Promotion start date convert........')

        promotion['end_date'] = pd.to_datetime(promotion['end_date'])
        logger.info('Promotion end date convert.........')

        promotion['duration'] = (promotion['end_date'] - promotion['start_date']).dt.days
        logger.info('Promotion calculate duration........')


        inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
        logger.info('Inventory snapahot date convert...........')

        product = product.apply(self.common_cleaning, axis=1)
        logger.info('Product common cleaning done.........')

        sales.to_csv(self.config.sales_t, index=False)
        logger.info('Sales Dataset saved........')

        promotion.to_csv(self.config.promotion_t, index=False)
        logger.info('Promotion dataset saved.........')

        inventory.to_csv(self.config.inventory_t, index=False)
        logger.info('Inventory dataset saved...........')

        product.to_csv(self.config.products_t, index=False)
        logger.info('Product datset saved.........')



In [25]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.load_data()
except Exception as e:
    raise e

[2026-04-01 00:20:14,965: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-01 00:20:14,968: INFO: common: created directory at: artifacts]
[2026-04-01 00:20:14,971: INFO: common: created directory at: artifacts/data_transformation]
[2026-04-01 00:20:15,565: INFO: 2564189832: Sales is concat Successful.....]
(53286, 31)
[2026-04-01 00:20:15,577: INFO: 2564189832: Sales filled the null values.......]


C:\Users\HP\AppData\Local\Temp\ipykernel_11744\2564189832.py:42: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sales['promo_type'].fillna('No Promo', inplace=True)
C:\Users\HP\AppData\Local\Temp\ipykernel_11744\2564189832.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

[2026-04-01 00:20:17,346: INFO: 2564189832: Sales datetime convert sucessful.......]
[2026-04-01 00:20:26,179: INFO: 2564189832: Sales Common cleaning done.........]
[2026-04-01 00:20:26,187: INFO: 2564189832: Promotion start date convert........]
[2026-04-01 00:20:26,195: INFO: 2564189832: Promotion end date convert.........]
[2026-04-01 00:20:26,199: INFO: 2564189832: Promotion calculate duration........]
[2026-04-01 00:20:26,213: INFO: 2564189832: Inventory snapahot date convert...........]
[2026-04-01 00:20:26,233: INFO: 2564189832: Product common cleaning done.........]
[2026-04-01 00:20:27,574: INFO: 2564189832: Sales Dataset saved........]
[2026-04-01 00:20:27,585: INFO: 2564189832: Promotion dataset saved.........]
[2026-04-01 00:20:27,615: INFO: 2564189832: Inventory dataset saved...........]
[2026-04-01 00:20:27,621: INFO: 2564189832: Product datset saved.........]
